In [ ]:
import time
import win32clipboard
import win32gui
import win32process
import win32api
import win32con
import psutil
import pyperclip
from pynput import keyboard

def get_active_window_info():
    """取得當前最前景視窗的標題與執行檔名稱"""
    hwnd = win32gui.GetForegroundWindow()
    if not hwnd:
        return "未知視窗", "未知程式"
    
    title = win32gui.GetWindowText(hwnd)
    try:
        _, pid = win32process.GetWindowThreadProcessId(hwnd)
        proc_name = psutil.Process(pid).name()
    except Exception:
        proc_name = "未知程式"
        
    return title, proc_name

def get_clipboard_content():
    """讀取剪貼簿內是檔案還是純文字"""
    copied_files = []
    try:
        win32clipboard.OpenClipboard()
        if win32clipboard.IsClipboardFormatAvailable(win32clipboard.CF_HDROP):
            data = win32clipboard.GetClipboardData(win32clipboard.CF_HDROP)
            if data:
                copied_files = list(data)
    except Exception:
        pass
    finally:
        try:
            win32clipboard.CloseClipboard()
        except Exception:
            pass

    if copied_files:
        return "FILES", copied_files
    else:
        return "TEXT", pyperclip.paste()

def send_key_combination(vk_code):
    """使用 Windows 原生 API 模擬按下 Ctrl + (C 或 V)"""
    # 釋放 Alt 鍵，避免影響 Ctrl+C/V
    win32api.keybd_event(win32con.VK_MENU, 0, win32con.KEYEVENTF_KEYUP, 0)
    time.sleep(0.05)
    
    # 按下 Ctrl -> 按下目標鍵 -> 釋放目標鍵 -> 釋放 Ctrl
    win32api.keybd_event(win32con.VK_CONTROL, 0, 0, 0)
    win32api.keybd_event(vk_code, 0, 0, 0)
    time.sleep(0.05)
    win32api.keybd_event(vk_code, 0, win32con.KEYEVENTF_KEYUP, 0)
    win32api.keybd_event(win32con.VK_CONTROL, 0, win32con.KEYEVENTF_KEYUP, 0)

def on_copy():
    """觸發 Ctrl + Alt + C"""
    print("\n" + "="*50)
    print("【偵測到 Ctrl + Alt + C：執行複製】")
    
    # 模擬標準 Ctrl+C
    send_key_combination(ord('C'))
    time.sleep(0.15)  # 等待剪貼簿寫入
    
    title, proc = get_active_window_info()
    content_type, content = get_clipboard_content()

    print(f"來源程式: {proc}")
    print(f"視窗標題: {title}")
    
    if content_type == "FILES":
        print(f"複製類型: 檔案/資料夾 (共 {len(content)} 個)")
        for idx, f in enumerate(content, 1):
            print(f"  {idx}. {f}")
    else:
        text_preview = content[:60] + ('...' if len(content) > 60 else '')
        print(f"複製類型: 純文字")
        print(f"複製內容: {text_preview}")
    print("="*50)

def on_paste():
    """觸發 Ctrl + Alt + V"""
    print("\n" + "="*50)
    print("【偵測到 Ctrl + Alt + V：執行貼上】")
    
    title, proc = get_active_window_info()
    content_type, content = get_clipboard_content()

    # 模擬標準 Ctrl+V
    send_key_combination(ord('V'))

    print(f"目標程式: {proc}")
    print(f"目標視窗: {title}")
    
    if content_type == "FILES":
        print(f"貼上類型: 檔案清單 ({len(content)} 個)")
    else:
        text_preview = content[:60] + ('...' if len(content) > 60 else '')
        print(f"貼上內容: {text_preview}")
    print("="*50)

def start_listener():
    print("【監聽啟動】請同時按住組合鍵：")
    print("  - Ctrl + Alt + C (複製並印出資訊)")
    print("  - Ctrl + Alt + V (貼上並印出資訊)")
    print("按 Ctrl + C (在命令提示字元視窗內) 可停止腳本\n")

    # 使用 GlobalHotKeys 原生組合鍵監聽，穩定度最高
    hotkeys = {
        '<ctrl>+<alt>+c': on_copy,
        '<ctrl>+<alt>+v': on_paste
    }

    with keyboard.GlobalHotKeys(hotkeys) as h:
        h.join()

if __name__ == "__main__":
    start_listener()

In [ ]:
from presidio_analyzer import (
    AnalyzerEngine, 
    EntityRecognizer, 
    RecognizerResult,
    PatternRecognizer,
    Pattern
)
from presidio_analyzer.nlp_engine import NlpEngineProvider
from transformers import pipeline

# ---------------------------------------------------------------------------
# 1. 建立 Presidio 的基礎中文 NLP 引擎 (解決 KeyError: 'zh')
# ---------------------------------------------------------------------------
# 告訴 Presidio 中文 ('zh') 語系使用 spaCy 的 zh_core_web_sm (如果沒有安裝會自動警告，預設可載入)
nlp_configuration = {
    "nlp_engine_name": "spacy",
    "models": [
        {"lang_code": "zh", "model_name": "zh_core_web_sm"},
        {"lang_code": "en", "model_name": "en_core_web_sm"}
    ],
}

provider = NlpEngineProvider(nlp_configuration=nlp_configuration)
custom_nlp_engine = provider.create_engine()

# ---------------------------------------------------------------------------
# 2. 自訂 Hugging Face 中文 NER 辨識器 (改用公開穩定的 ckiplab 模型)
# ---------------------------------------------------------------------------
class CustomHfChineseRecognizer(EntityRecognizer):
    def __init__(self):
        super().__init__(
            supported_entities=["PERSON", "LOCATION", "ORGANIZATION"],
            supported_language="zh"
        )
        print("正在載入 Hugging Face 中文 NER 模型 (ckiplab/bert-base-chinese-ner)...")
        
        # 改用確定公開可下載的 CKIP 繁體/通用中文 NER 模型
        self.ner_pipeline = pipeline(
            "ner", 
            model="ckiplab/bert-base-chinese-ner", 
            aggregation_strategy="simple"
        )

    def load(self):
        pass

    def analyze(self, text, entities, nlp_artifacts=None):
        results = []
        ner_results = self.ner_pipeline(text)
        
        # ckiplab 的標籤對映：PER -> PERSON, LOC -> LOCATION, ORG -> ORGANIZATION
        label_map = {
            "PER": "PERSON", 
            "LOC": "LOCATION", 
            "ORG": "ORGANIZATION"
        }

        for item in ner_results:
            entity_type = label_map.get(item["entity_group"])
            if entity_type and (not entities or entity_type in entities):
                results.append(
                    RecognizerResult(
                        entity_type=entity_type,
                        start=item["start"],
                        end=item["end"],
                        score=float(item["score"])
                    )
                )
        return results
# ---------------------------------------------------------------------------
# 3. 初始化 AnalyzerEngine 並註冊組件
# ---------------------------------------------------------------------------
# 傳入剛剛配置好的 custom_nlp_engine，並明確聲明支援 ["zh", "en"]
analyzer = AnalyzerEngine(
    nlp_engine=custom_nlp_engine, 
    supported_languages=["zh", "en"]
)

# 掛載自訂的 Hugging Face 模型
analyzer.registry.add_recognizer(CustomHfChineseRecognizer())


# ---------------------------------------------------------------------------
# 4. 測試執行
# ---------------------------------------------------------------------------
text = "你好，我是張偉，我的信箱是 zhangwei@example.com，我在台積電上班。"
results = analyzer.analyze(text=text, language="zh")

print("\n--- 檢測結果 ---")
for res in results:
    print(f"實體: {res.entity_type:<15} | 文字: {text[res.start:res.end]:<15} | 信心度: {res.score:.2f}")

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForTokenClassification

text = "本案理賠申請人為王志明（英文名 Chih-Ming Wang），身分證字號為 D287654321"
model_name = "uer/roberta-base-finetuned-cluener2020-chinese"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

# 1. 將文字轉換為 Token IDs
inputs = tokenizer(text, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

# 2. 模型推論取得算分 (Logits)
with torch.no_grad():
    outputs = model(**inputs)

# 3. 使用 Softmax 算出每個 Token 對應各標籤的機率 (信心度)
probs = F.softmax(outputs.logits[0], dim=-1)
max_probs, pred_indices = torch.max(probs, dim=-1)

# 4. 取得標籤對照表 (id -> label)
id2label = model.config.id2label

print(f"原始句子: {text}\n" + "=" * 65)
print(f"{'Token (字/詞)':<12} | {'預測標籤':<12} | {'信心度 (Score)':<10}")
print("-" * 65)

# 5. 印出每一個 Token (排除開頭 [CLS] 與結尾 [SEP])
for i in range(1, len(tokens) - 1):
    token = tokens[i]
    label = id2label[pred_indices[i].item()]
    score = max_probs[i].item()
    print(f"{token}/",end='')
print()
for i in range(1, len(tokens) - 1):
    token = tokens[i]
    label = id2label[pred_indices[i].item()]
    score = max_probs[i].item()
    print(f"{label}/",end='')

原始句子: 本案理賠申請人為王志明（英文名 Chih-Ming Wang），身分證字號為 D287654321
Token (字/詞)  | 預測標籤         | 信心度 (Score)
-----------------------------------------------------------------
本/案/理/賠/申/請/人/為/王/志/明/（/英/文/名/chi/##h/-/ming/wang/）/，/身/分/證/字/號/為/d2/##87/##65/##43/##21/
O/O/O/O/O/O/O/O/B-name/I-name/I-name/O/O/O/O/B-name/I-name/I-name/I-name/I-name/O/O/O/O/O/O/O/O/O/O/O/O/O/

In [21]:
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider

# 1. 初始化 Presidio 預設的 NlpEngine (底層使用 spaCy)
provider = NlpEngineProvider(nlp_configuration={
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "en", "model_name": "en_core_web_sm"}]
})
nlp_engine = provider.create_engine()

text = "本案理賠申請人為王志明（英文名  Chih-Ming Wang），身分證字號為 D287654321"

# 2. 讓 Presidio 的 NLP 引擎先對文字進行處理 (包含語法分析與斷詞)
nlp_artifacts = nlp_engine.process_text(text, language="en")

print(f"原始輸入文字: {text}\n" + "="*60)
print(f"{'Presidio/spaCy 斷詞':<18} | {'詞性標籤 (POS)':<12} | {'是否為 PII 關鍵字'}")
print("-" * 60)

# 3. 取得 Presidio 識別出的 PII 區段位置
analyzer = AnalyzerEngine(nlp_engine=nlp_engine)
pii_results = analyzer.analyze(text=text, language="en")
pii_spans = [(r.start, r.end, r.entity_type, r.score) for r in pii_results]

# 4. 印出全句的每一個 Token
for token in nlp_artifacts.tokens:
    # 檢查該 Token 是否屬於 Presidio 抓出的 PII
    is_pii = "否"
    for start, end, entity_type, score in pii_spans:
        if start <= token.idx < end:
            is_pii = f"是 ({entity_type}, 信心度:{score:.2f})"
            break
            
    print(f"{token.text:<20} | {token.pos_:<14} | {is_pii}")
# 本案理賠申請人為王志明（英文名 /Chih/-/Ming /Wang），身分證字號為 D287654321
# PROPN                      /PERSON/PUNCT/PERSON/PROPN                  

原始輸入文字: 本案理賠申請人為王志明（英文名  Chih-Ming Wang），身分證字號為 D287654321
Presidio/spaCy 斷詞  | 詞性標籤 (POS)   | 是否為 PII 關鍵字
------------------------------------------------------------
本案理賠申請人為王志明（英文名      | PROPN          | 是 (PERSON, 信心度:0.85)
                     | SPACE          | 是 (PERSON, 信心度:0.85)
Chih                 | PROPN          | 是 (PERSON, 信心度:0.85)
-                    | PUNCT          | 是 (PERSON, 信心度:0.85)
Ming                 | PROPN          | 是 (PERSON, 信心度:0.85)
Wang），身分證字號為         | PROPN          | 否
D287654321           | PROPN          | 是 (US_DRIVER_LICENSE, 信心度:0.30)


In [20]:
import spacy

nlp = spacy.load("en_core_web_sm")  # 跟你程式碼用的 en_core_web_trf 是同系列,只是精度/速度不同
text = "本案理賠申請人為王志明（英文名 : Chih-Ming Wang），身分證字號為 D287654321"

doc = nlp(text)

print("=== spaCy 原始 NER 結果 ===")
for ent in doc.ents:
    print(f"文字: {ent.text!r:30} 標籤: {ent.label_:10} 位置: [{ent.start_char}, {ent.end_char}]")
from presidio_analyzer import AnalyzerEngine

analyzer = AnalyzerEngine()  # 這裡預設也是吃 en_core_web_lg,你可以改成用你自訂的 registry
results = analyzer.analyze(text=text, language="en")

print("\n=== Presidio 轉換後結果 ===")
for r in results:
    print(f"標籤: {r.entity_type:10} 位置: [{r.start}, {r.end}]  信心分數: {r.score}")

=== spaCy 原始 NER 結果 ===

=== Presidio 轉換後結果 ===
標籤: PERSON     位置: [18, 27]  信心分數: 0.85
標籤: US_DRIVER_LICENSE 位置: [41, 51]  信心分數: 0.3


In [3]:
# download_model.py (請在有網路的電腦執行)
import os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 推薦使用 Opus-MT 的英中翻譯模型
MODEL_NAME = "Helsinki-NLP/opus-mt-en-zh"
SAVE_DIR = "./offline_model"

print("正在下載離線翻譯模型與分詞器...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# 將模型完整存入本地資料夾
tokenizer.save_pretrained(SAVE_DIR)
model.save_pretrained(SAVE_DIR)

print(f"✅ 模型下載完成！請打包【{os.path.abspath(SAVE_DIR)}】資料夾。")

正在下載離線翻譯模型與分詞器...


d:\2.programm2\github-star\presidio-research\venvpr\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
d:\2.programm2\github-star\presidio-research\venvpr\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\Tools\huggingface_cache\hub\models--Helsinki-NLP--opus-mt-en-zh. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, se

✅ 模型下載完成！請打包【d:\2.programm2\topic\presidio-application\test\offline_model】資料夾。


In [1]:
# translate_offline.py (請在無網路的 Windows 電腦執行)
import os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from opencc import OpenCC

# 指向離線模型的本地資料夾路徑
MODEL_DIR = "./offline_model"

def load_offline_translator():
    if not os.path.exists(MODEL_DIR):
        raise FileNotFoundError(f"找不到離線模型資料夾: {MODEL_DIR}")
        
    print("⏳ 正在載入本地離線翻譯模型...")
    # 強制開啟 local_files_only，避免嘗試連線網路
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR, local_files_only=True)
    
    # 初始化簡轉繁轉換器 (s2twp: 簡體轉台灣繁體，含慣用語修正)
    cc = OpenCC('s2twp')
    
    return tokenizer, model, cc

def translate_en_to_zhtw(text, tokenizer, model, cc):
    # 分詞與特徵提取
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    
    # 進行模型推論生成翻譯
    translated_tokens = model.generate(**inputs)
    
    # 解碼成文字 (簡體中文)
    zh_cn_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    
    # 本地轉為繁體中文
    zh_tw_text = cc.convert(zh_cn_text)
    return zh_tw_text

if __name__ == "__main__":
    tokenizer, model, cc = load_offline_translator()
    print("✨ 離線翻譯系統已就緒！請輸入英文段落（輸入 q 離開）：\n")
    
    while True:
        english_text = input("EN > ").strip()
        if english_text.lower() == 'q':
            break
        if not english_text:
            continue
            
        result = translate_en_to_zhtw(english_text, tokenizer, model, cc)
        print(f"繁中 > {result}\n")

d:\2.programm2\github-star\presidio-research\venvpr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ 正在載入本地離線翻譯模型...


d:\2.programm2\github-star\presidio-research\venvpr\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


✨ 離線翻譯系統已就緒！請輸入英文段落（輸入 q 離開）：

繁中 > _____________________________________________________________________________________________________________________________________________________________________________________________

